# **Sesión 2:** Calibración de Cámara 📷⚙️

## 0. Preparación

In [2]:
import cv2
print("OpenCV should be 4.8.x Current version:", cv2.__version__)
from typing import List
import numpy as np
# import imageio --- IGNORE IF DEPRECATED---
import imageio.v3 as imageio
import copy
import os
import glob

OpenCV should be 4.8.x Current version: 4.8.0


## **Apartado A: Calibración de cámara** (derecha e izquierda)

En este apartado se realiza la calibración de dos cámaras de un sistema estereoscópico. Para ello se trabajará con las imágenes de las carpetas ``left`` y ``right``. En primer lugar se trabajará con la carpeta ``left``. Posteriormente, deberá repetir el proceso con las imágenes en la carpeta ``right``. Ambas carpetas contienen imágenes con las que se calibrarán las cámaras. En ellas aparece el patrón de calibración en diferentes posiciones y orientaciones. Estas imágenes serán los datos de entrada.

Los pasos que deberá seguir para calibrar una cámara son:

1. Defina y ejecute el método para cargar imágenes ``load_images()``.
2. Detecte las esquinas de los patrones usando ``cv2.findChessboardCorners()``. Refine las detecciones con ``cv2.cornerSubPix()``.
3. Compruebe que las detecciones son correctas dibujando los resultados con ``cv2.drawChessboardCorners()``.
4. Defina y ejecute el método ``get_chessboard_points(chessboard_shape, dx, dy)`` que proporcione las coordenadas 3D de las esquinas del patrón. El sistema de referencia utilizado deberá estar anclado al propio patrón.
5. Utilice ``cv2.calibrateCamera`` para obtener los parámetros de calibración para la cámara izquierda.

### **Tarea A.1:** Defina y ejecute el método para cargar imágenes ``load_images()``.

In [3]:
%pip install pillow-heif

Note: you may need to restart the kernel to use updated packages.


In [4]:
from PIL import Image
from pillow_heif import register_heif_opener
import glob, os

register_heif_opener()
for f in glob.glob("data/*.HEIC") + glob.glob("data/*.heic"):
    Image.open(f).convert("RGB").save(os.path.splitext(f)[0] + ".jpg", quality=95)
print("Convertidas")

Convertidas


In [5]:
def load_images(filenames: List) -> List:
    return [cv2.imread(filename) for filename in filenames]

In [6]:
# TODO Build a list containing the paths of all images from the left camera
imgs_path = sorted(glob.glob("data/*.jpg"))
imgs = load_images(imgs_path)
print(f"Loaded {len(imgs)} images")

Loaded 56 images


### **Tarea A.2:** Detecte las esquinas de los patrones usando ``cv2.findChessboardCorners()``. Refine las detecciones con ``cv2.cornerSubPix()``.

In [7]:
# TODO Find corners with cv2.findChessboardCorners()
# Hint: you can use a list comprehension to process all images
chessboard_shape = (6, 8)  
corners = [cv2.findChessboardCorners(img, chessboard_shape) for img in imgs]
print(f"Correct detections: {sum(result[0] for result in corners)}/{len(corners)}")

Correct detections: 56/56


In [8]:
corners_copy = copy.deepcopy(corners)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)

# TODO To refine corner detections with cv2.cornerSubPix() you need to input grayscale images. Build a list containing grayscale images.
imgs_gray = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs]

corners_refined = []
for gray, (found, detected_corners) in zip(imgs_gray, corners_copy):
    if found:
        refined = cv2.cornerSubPix(gray, detected_corners, (3, 3), (-1, -1), criteria)
        corners_refined.append((found, refined))
    else:
        corners_refined.append((found, detected_corners))


### **Tarea A.3:** Compruebe que las detecciones son correctas dibujando los resultados con ``cv2.drawChessboardCorners()``

In [9]:
imgs_copy = copy.deepcopy(imgs)

In [10]:
# TODO Use cv2.drawChessboardCorners() to draw the cornes
for i, (found, refined) in enumerate(corners_refined):
    if found:
        cv2.drawChessboardCorners(imgs_copy[i], chessboard_shape, refined, found)

In [11]:
# TODO Show images and save when needed

def show_image(window_name, image, scale=0.25):
    small = cv2.resize(image, None, fx=scale, fy=scale)
    cv2.imshow(window_name, small)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def write_image(filename, image):
    cv2.imwrite(filename, image)

# Mostrar y guardar las imágenes con las esquinas dibujadas
os.makedirs("data/left_corners", exist_ok=True)
for i, (found, _) in enumerate(corners_refined):
    if found:
        write_image(f"data/left_corners/corners_{i:02d}.jpg", imgs_copy[i])

show_image("Left camera - image 1", imgs_copy[0])


### **Tarea A.4:** Defina y ejecute el método ``get_chessboard_points(chessboard_shape, dx, dy)``

In [12]:
# TODO Design the method. It should return a np.array with np.float32 elements
def get_chessboard_points(chessboard_shape, dx, dy):
    cols, rows = chessboard_shape
    points = np.zeros((cols * rows, 3), dtype=np.float32)
    points[:, :2] = np.mgrid[0:cols, 0:rows].T.reshape(-1, 2)
    points[:, 0] *= dx
    points[:, 1] *= dy
    return points

In [13]:
square_size = 25  # mm (compruébalo con regla en el papel)
chessboard_points = [get_chessboard_points(chessboard_shape, square_size, square_size)
                     for found, _ in corners_refined if found]

### **Tarea A.5:** Utilice ``cv2.calibrateCamera()`` para obtener los parámetros de calibración para la cámara izquierda

In [14]:
# Filter data and get only those with adequate detections
valid_corners = [refined for found, refined in corners_refined if found]
# Convert list to numpy array with np.float32 elements
valid_corners = [np.asarray(c, dtype=np.float32) for c in valid_corners]
print(f"Images used for calibration: {len(valid_corners)}")

Images used for calibration: 56


In [15]:
# Comprobar la resolución de las imágenes
print("Tamaño de imagen (alto, ancho):", imgs_gray[0].shape)

# Calibración (sin k3 para que la distorsión no se dispare)
rms, intrinsics, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
    chessboard_points, valid_corners, imgs_gray[0].shape[::-1], None, None,
    flags=cv2.CALIB_FIX_K3
)

# Obtain extrinsics
extrinsics = list(map(lambda rvec, tvec: np.hstack((cv2.Rodrigues(rvec)[0], tvec)), rvecs, tvecs))

Tamaño de imagen (alto, ancho): (3024, 4032)


In [16]:
# Print outputs
print("Intrinsics:\n", intrinsics)
print("Distortion coefficients:\n", dist_coeffs)
print("Root mean squared reprojection error:\n", rms)

Intrinsics:
 [[2.79400152e+03 0.00000000e+00 2.06368564e+03]
 [0.00000000e+00 2.79353716e+03 1.41362620e+03]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Distortion coefficients:
 [[ 0.09675499 -0.0062489  -0.0198398   0.00159888  0.        ]]
Root mean squared reprojection error:
 1.8795148782996536


In [17]:
np.savez("calibracion.npz", K=intrinsics, dist=dist_coeffs, rms=rms)